In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = r"D:\DATA\abmil_exp3.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 2, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 3,
    'Resection Margin Not Free': 3, 
    'Proliferative/Pre-neoplastic Changes': 3, 
    'Benign Neoplasm': 3, 
    'Uncertain / Borderline Neoplasm': 3, 
    'In Situ Neoplasm': 3, 
    'Malignant Neoplasm': 3,
}

df_all["M_idx"] = df_all["M_category"].apply(
    lambda lst: [class_dict[x] for x in lst]
)

# 0: Normal
# 1: Other morphologies
# 2: Inflammation
# 3: Neoplastic Changes, Benign/Uncertain/Borderline, In Situ, Malignant Neoplasm 

In [ ]:
df_sub = df_all.copy()
df_sub['M_idx'] = df_sub['M_idx'].apply(
    lambda x: max(x) if isinstance(x, list) else x
)

In [ ]:
df_sub['M_idx'].value_counts()

In [ ]:
from abmil import TrainABMILPipeline

save_path = r"D:\NOTEBOOKS\Christine\abmil_checkpoints\abmil_hopt_full.pt"

pipeline = TrainABMILPipeline(df_sub, "filename", "M_idx", "features_h-optimus-0", "tiles_224", zarr_dir, save_path)
pipeline.validate_slides()

In [ ]:
# Determine best epoch

# Training hyperparameters
max_tiles = 50000  # Limit tiles per slide to avoid memory issues
n_epochs = 100  # Max epochs (early stopping may stop sooner)
seed = 42
validation_fraction = 0.10  # 90/10 train/val split
early_stopping_patience = 5  # Stop if no AUC improvement for N epochs

# Train on 90/10 split
model, best_epoch = pipeline.train_abmil(
    max_tiles=max_tiles,
    n_epochs=n_epochs,
    seed=seed,
    validation_fraction=validation_fraction,
    early_stopping_patience=early_stopping_patience,
)

print(f"Best epoch found: {best_epoch}")
print(f"Model device: {next(model.parameters()).device}")

In [ ]:
# Retrain on all data, using best epoch
model, _, _ = pipeline.save_abmil(max_tiles=50000, n_epochs=18, seed=42)

In [ ]:
from scripts.abmil import load_checkpoint

# Load and verify checkpoint
loaded_model, config, loaded_label_mapping = load_checkpoint(checkpoint_path)

print(f"\nConfig keys: {config.keys()}")
print(f"Config:\n{config}")
print(f"\nLabel mapping: {loaded_label_mapping}")

In [ ]:
import os
import numpy as np
import pandas as pd
from wsidata import open_wsi

feature_keys = {
    "H-optimus-0": "features_h-optimus-0"
}

paths = df_sub["filename"].tolist()
label_map = df_sub.groupby("filename")["M_idx"].max().to_dict()

rows = []

for slide_path in paths:
    print(slide_path)
    zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))

    try:
        wsi = open_wsi(slide_path, zarr_path)
    except Exception as e:
        print(e)
        continue

    label = label_map.get(slide_path)

    for model, key in feature_keys.items():
        X = wsi.tables.get(key, {}).X if key in wsi.tables else None
        if X is None or X.size == 0:
            continue

        norms = np.linalg.norm(X, axis=1)
        bag_size = X.shape[0]

        rows.extend([
            {
                "filename": slide_path,
                "model": model,
                "M_idx": label,
                "bag_size": bag_size,
                "feature_norm": float(n),
            }
            for n in norms
        ])

df_plot = pd.DataFrame(rows)

In [ ]:
print(df_plot.head())

In [ ]:
norm_summary = (
    df_plot
    .groupby("model")["feature_norm"]
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

print("Norm Summary:",
      norm_summary)

bag_df = (df_plot[df_plot["model"] == "H-optimus-0"])

bag_summary = (
    bag_df
    .groupby("M_idx")["bag_size"]
    .agg(["count", "mean", "std", "median"])
    .round(2)
)

print("Bag Summary:",
      bag_summary)

In [ ]:
m_idx_map = {
    0: "0 - Normal (n=121)",
    1: "1 - Other (n=223)",
    2: "2 - Inflammation (n=227)",
    3: "3 - Neoplasms (n=429)"
}

bag_df["M_label"] = bag_df["M_idx"].map(m_idx_map)

In [ ]:
# Plot, bagsize by M idx

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

order = ["0 - Normal (n=121)", "1 - Other (n=223)", "2 - Inflammation (n=227)", "3 - Neoplasms (n=429)"]

palette = sns.color_palette(n_colors=len(order))
color_map = dict(zip(order, palette))

sns.set_style("white")

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=bag_df,
    x="bag_size",
    hue="M_label",
    hue_order=order,
    bins=50,
    multiple="stack",
    alpha=0.4,
    palette=color_map,
    ax=ax
)

legend = ax.get_legend()
legend.set_title("M idx")

handles = legend.legend_handles
ymax = ax.get_ylim()[1]

for i, label in enumerate(order):
    values = bag_df.loc[bag_df["M_label"] == label, "bag_size"]

    mean_val = values.mean()
    median_val = values.median()
    color = color_map[label]

    ax.axvline(mean_val, color=color, linestyle="-", linewidth=2)
    ax.axvline(median_val, color=color, linestyle="--", linewidth=2)

    # Mean
    ax.text(
        mean_val,
        ymax * 0.9,
        f"Mean: {mean_val:.1f}",
        color=color,
        ha='left',
        fontsize=8, 
        backgroundcolor='white'
    )
    
    # Median
    ax.text(
        median_val,
        ymax * 0.75,
        f"Median: {median_val:.1f}",
        color=color,
        ha='left',
        fontsize=8,
        fontweight='bold',
        backgroundcolor='white'
    )

ax.set_title("Bag Size Distribution")
ax.set_xlabel("Instances per Slide")
ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
df_plot["norm_zscore"] = df_plot.groupby(["filename", "model"])["feature_norm"] \
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- (1) Raw norms ---
sns.histplot(
    data=df_plot,
    x="feature_norm",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[0],
    legend= True
)
legend = ax.get_legend()
legend.set_title("M idx")
axes[0].set_title("Feature Norm Distribution")
axes[0].set_xlabel("L2 Norm")

# --- (2) Normalized norms ---
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[1],
)
axes[1].set_title("Normalized Norms (Per-Slide)")
axes[1].set_xlabel("Z-scored Norm")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate mean feature norm per slide
df_agg = df_plot[df_plot["model"] == "H-optimus-0"].groupby("filename").agg({
    "feature_norm": "mean",
    "bag_size": "first",
    "M_idx": "first"
}).reset_index()

# Map M_idx to labels
m_idx_map = {
    0: "0 - Normal",
    1: "1 - Other",
    2: "2 - Inflammation",
    3: "3 - Neoplasms"
}
df_agg["M_label"] = df_agg["M_idx"].map(m_idx_map)

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_agg,
    x="bag_size",
    y="feature_norm",
    hue="M_label",
    s=100,
    alpha=0.6,
    ax=ax
)

ax.set_xlabel("Bag Size")
ax.set_ylabel("Mean Feature Norm (L2)")
ax.set_title("Mean Feature Norm vs. Bag Size by M idx")
ax.legend(title="Morphology Category", bbox_to_anchor=(1.05, 1), loc='upper left')

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
from visualize_features_new import FeatureDataBuilder

cache_file = r"D:\NOTEBOOKS\Christine\all_slides\cache_tissue_artifact_new.pkl"
models = ["h-optimus-0"]
all_filenames = df_sub["filename"].tolist()

featurebuilder = FeatureDataBuilder(all_filenames, df_sub, zarr_dir, cache_file, models)
df = featurebuilder.df_merged

In [ ]:
from visualize_features import FeatureVisualizer

categories = ["T_category", "T_text", "M_category", "M_text", "team", "sex", "alder", "alder gruppe", 'mattype tekst', 'stain', 'snomed_code',
              'snomed_text', 'undersoeger_anonymous', 'M_idx'] 

for model in models:
    feat_col = f"features_{model}"
    for cat in categories:
        try:
            print(f"\n Visualizing model: {model}, feature: {cat}")
            viz = FeatureVisualizer(df, label_col = cat, features_col = feat_col, artifact_col = "artifact_default_pct")
            viz.print_score()
            viz.tsne_plot()
            
        except Exception as e:
            print(f"Error visualizing projection: {e}")
            continue